# M2 Agentic AI - ATS Resume Tailoring (Prompt Patterns Version)

We’re excited to have you here in the **Agentic AI** course! In this ungraded lab, you’ll reuse the same **reflection pattern** from the chart-generation notebook, but apply it to a resume-tailoring task.

## 1. Introduction
### 1.1. Lab overview

In this lab, the workflow accepts two strings:

- `resume_text`: the candidate's existing resume
- `vacancy_text`: the target job description

The agent first analyzes the vacancy for ATS-relevant keywords, generates a tailored resume **V1**, reflects on V1 against both source documents, and produces an improved **V2**.

The key constraint is **truthfulness**: the workflow may improve wording, ordering, and keyword alignment, but it must never invent skills, employers, achievements, dates, certifications, or metrics that are not supported by the original resume.

The workflow mirrors the original notebook pattern:

**Generate → Parse/Execute → Reflect → Regenerate → Compare**.


## 2. Setup: Initialize environment and client

As in the original notebook, you will use:

- **`re`** for tag extraction and lightweight keyword matching.
- **`json`** for structured LLM responses.
- **`pandas` / `matplotlib`** for ATS diagnostics and visualizations.
- **`utils`** as the existing helper module for model calls and notebook display.

The LLM calls continue to go through `utils.get_response(...)`, so the notebook keeps the same client abstraction as the original lab. In this version, `utils.get_response(...)` is backed by the Cursor CLI and authenticates with `CURSOR_API_KEY`, so no `openai` Python package is required.


In [ ]:
# Standard library imports
import re
import json

# Third-party imports used for diagnostics / charts
import pandas as pd
import matplotlib.pyplot as plt

# Local helper module
import utils


In [ ]:
# Cursor CLI / authentication check
import os
import shutil

assert os.getenv("CURSOR_API_KEY"), "CURSOR_API_KEY is not set in the environment or .env file"
assert shutil.which("agent"), "Cursor CLI is not installed or 'agent' is not on PATH"

print("Cursor CLI:", shutil.which("agent"))
print("CURSOR_API_KEY: configured")


### 2.1. Resume and vacancy inputs

Instead of loading a CSV dataset, this version of the workflow works directly with two strings. Replace the examples below with your real resume and vacancy text.


In [ ]:
resume_text = """
Alex Morgan
Data Analyst

EXPERIENCE
Data Analyst, Example Retail
- Built Python and SQL reports for weekly sales analysis.
- Automated recurring Excel reporting and reduced manual reporting time by 30%.
- Created dashboards for business stakeholders.

SKILLS
Python, SQL, Excel, Data Analysis, Reporting, Data Visualization
""".strip()

vacancy_text = """
We are hiring a Data Analyst to analyze business performance and build reliable reporting.
The successful candidate will use Python, SQL, Excel, dashboards, and data visualization.
Experience with Power BI and stakeholder communication is preferred.
Knowledge of AWS is a plus.
""".strip()

utils.print_html(resume_text, title="Original Resume")
utils.print_html(vacancy_text, title="Target Vacancy")


You’ll now build the same kind of agentic workflow as in the original notebook. The “artifact” being improved is no longer chart code; it is a tailored resume.


## 3. Building the pipeline


### 3.1 Step 1 — Analyze the vacancy using the resume prompt patterns

Before rewriting the resume, the first LLM call combines five of the provided prompt patterns while preserving the same single-step structure used in the original notebook:

- **KEYWORDS** — identify the ATS terms and phrases most likely to matter.
- **ROLE PLAY (Career Advisor)** — identify the five areas the candidate should emphasize for this role.
- **SKILL COMPARISON** — compare vacancy requirements with evidence in the original resume.
- **SKILL MATCH** — identify which existing skills and experiences should be highlighted more strongly.
- **ACTION VERBS** — suggest role-appropriate action verbs that can truthfully describe existing experience.

This is still one analysis function, not five independent agents. The goal is to enrich the structured analysis passed into the same Generate → Reflect pattern.

A missing vacancy keyword remains a **gap** and must not be inserted into the resume without evidence.

In [ ]:
def analyze_ats_match(resume_text: str, vacancy_text: str, model: str) -> dict:
    """Analyze ATS fit using the provided resume prompt patterns and source evidence."""

    prompt = f"""
    You are acting as both an ATS resume analyst and a career advisor.

    JOB DESCRIPTION:
    {vacancy_text}

    ORIGINAL RESUME:
    {resume_text}

    Apply the following resume prompt patterns in one structured analysis.

    1. KEYWORDS
    Analyze the job description and identify the most important ATS keywords and phrases
    that an applicant tracking system may track.

    2. ROLE PLAY (Career Advisor)
    For this target role, identify the five most important things a career advisor would
    recommend emphasizing in the candidate's resume.

    3. SKILL COMPARISON
    Compare the skills and experience requested in the vacancy with evidence available
    in the ORIGINAL RESUME.

    4. SKILL MATCH
    Identify which existing candidate skills and experiences should be highlighted more
    strongly for this specific position.

    5. ACTION VERBS
    Suggest strong action verbs appropriate for the target role, but only verbs that could
    truthfully describe work already supported by the ORIGINAL RESUME.

    Return a single VALID JSON object only. Do not use Markdown or code fences.

    Required JSON schema:
    {{
      "target_role": "",
      "career_advisor_priorities": [
        {{"priority": "", "reason": ""}}
      ],
      "recommended_action_verbs": [],
      "skills_to_highlight": [],
      "top_keywords": [
        {{
          "keyword": "",
          "importance": "required|preferred|context",
          "status": "matched|partial|missing",
          "evidence": "supporting evidence from the original resume, or empty string when missing",
          "resume_opportunity": "how this supported keyword/skill could be emphasized, or empty string if missing"
        }}
      ],
      "required_skills": [],
      "preferred_skills": [],
      "tools_and_technologies": [],
      "soft_skills": [],
      "experience_requirements": [],
      "education_requirements": []
    }}

    RULES:
    1. Select at most 20 meaningful ATS keywords or phrases from the job description.
    2. Mark a keyword as "matched" only when the ORIGINAL RESUME clearly supports it.
    3. Use "partial" when the resume contains related evidence but not the exact requirement.
    4. Use "missing" when there is no supporting evidence in the ORIGINAL RESUME.
    5. Every matched or partial keyword must include concrete supporting evidence from the ORIGINAL RESUME.
    6. Never infer or invent candidate experience, employers, skills, technologies, education, certifications, metrics, or achievements.
    7. Preserve important employer terminology when useful for ATS matching.
    8. Do not treat generic words such as "work", "team", or "company" as ATS keywords by themselves.
    9. Career-advisor priorities and skills_to_highlight must be based on the intersection between the vacancy and the ORIGINAL RESUME.
    10. recommended_action_verbs are suggestions for wording only; they must not imply responsibilities or achievements unsupported by the ORIGINAL RESUME.
    """

    response = utils.get_response(model, prompt).strip()

    try:
        return json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*\}", response)
        if not match:
            raise ValueError("The model did not return a valid ATS JSON object.")
        return json.loads(match.group(0))

Now run the ATS analysis and inspect the structured output.


In [ ]:
ats_analysis = analyze_ats_match(
    resume_text=resume_text,
    vacancy_text=vacancy_text,
    model="auto",
)

utils.print_html(
    json.dumps(ats_analysis, indent=2, ensure_ascii=False),
    title="ATS Analysis"
)


The important part of this step is the `status` + `evidence` pair. A keyword marked `missing` is a real gap and must **not** be added to the resume just to improve apparent ATS coverage.


### 3.2. Step 2 — Generate the tailored resume (V1)

This is the direct analogue of generating the first artifact draft in the original notebook.

The generation prompt now explicitly applies the provided **REVISE**, **ACTION VERBS**, **KEYWORDS**, **SKILL COMPARISON**, and **SKILL MATCH** patterns using the structured analysis from Step 1.

The model still returns the resume between `<resume>...</resume>` tags, so extraction and downstream reflection follow the same notebook pattern.

In [ ]:
def generate_resume(
    resume_text: str,
    vacancy_text: str,
    ats_analysis: dict,
    model: str,
) -> str:
    """Generate ATS-tailored Resume V1 using the supplied prompt patterns."""

    prompt = f"""
    You are an expert resume writer specializing in Applicant Tracking Systems.

    ORIGINAL RESUME:
    {resume_text}

    JOB DESCRIPTION:
    {vacancy_text}

    ATS / CAREER ADVISOR ANALYSIS:
    {json.dumps(ats_analysis, indent=2, ensure_ascii=False)}

    Apply the following resume prompt patterns when producing Resume V1.

    ACTION VERBS:
    Use strong active verbs appropriate for the target role when they truthfully describe
    experience already present in the ORIGINAL RESUME. Prefer useful verbs from
    recommended_action_verbs where appropriate, but do not force them.

    KEYWORDS:
    Naturally incorporate important ATS keywords and phrases from top_keywords only when
    their status is "matched" or "partial" and the ORIGINAL RESUME contains supporting evidence.

    REVISE:
    Improve experience bullets for this vacancy. Where evidence allows, use the structure:
    Action + Task + Method/Skill + Result.
    Preserve existing quantified results and do not invent new metrics.

    SKILL COMPARISON:
    Use the comparison between vacancy requirements and resume evidence to decide which
    supported skills belong in the experience descriptions and skills section.

    SKILL MATCH:
    Prioritize the existing skills and experiences identified in skills_to_highlight and
    career_advisor_priorities when they are relevant to the target position.

    Return your answer STRICTLY in this format:

    <resume>
    tailored resume here
    </resume>

    Do not add explanations outside the tags.

    HARD RULES:
    1. Use only facts supported by the ORIGINAL RESUME.
    2. NEVER invent employers, responsibilities, skills, achievements, numbers,
       technologies, certifications, dates, education, or job titles.
    3. Do not add top_keywords whose status is "missing".
    4. A "partial" keyword may be used only in wording that accurately reflects the existing evidence; do not upgrade partial evidence into full expertise.
    5. Prefer terminology from the job description when it truthfully describes existing experience.
    6. Preserve measurable achievements already present in the ORIGINAL RESUME.
    7. Prioritize the most vacancy-relevant experience and skills.
    8. Avoid keyword stuffing and unnecessary repetition.
    9. Keep ATS-friendly formatting: plain headings, simple bullets, no tables, columns, icons, or graphics.
    10. The candidate must be able to explain and defend every line in an interview.
    """

    return utils.get_response(model, prompt)


def extract_resume(tagged_resume: str) -> str:
    """Extract resume text from <resume>...</resume> tags."""
    match = re.search(r"<resume>([\s\S]*?)</resume>", tagged_resume)
    if not match:
        raise ValueError("No <resume>...</resume> block found in model output.")
    return match.group(1).strip()

In [ ]:
# Generate initial tailored resume
resume_v1_tagged = generate_resume(
    resume_text=resume_text,
    vacancy_text=vacancy_text,
    ats_analysis=ats_analysis,
    model="auto",
)

utils.print_html(resume_v1_tagged, title="LLM Output with First Draft Resume (V1)")

# Extract the text within the <resume> tags
resume_v1 = extract_resume(resume_v1_tagged)
utils.print_html(resume_v1, title="Extracted Resume (V1)")


### 3.3. Step 3 — Calculate transparent ATS diagnostics

Instead of asking the model to invent an opaque “ATS score,” this notebook calculates simple, explainable metrics from the vacancy keywords.

Two coverage metrics are shown:

- **Vacancy keyword coverage**: how many analyzed vacancy keywords appear in the generated resume.
- **Supported keyword usage**: how many keywords supported by the original resume are actually used in the generated resume.

The workflow also checks whether any keywords classified as `missing` were introduced into the generated resume. That is treated as an **unsupported-keyword risk**, not as a positive score.


In [ ]:
def _normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#./ -]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _keyword_present(keyword: str, text: str) -> bool:
    keyword_norm = _normalize_text(keyword)
    text_norm = _normalize_text(text)
    if not keyword_norm:
        return False
    return keyword_norm in text_norm


def calculate_ats_metrics(resume_text: str, ats_analysis: dict) -> dict:
    """Calculate transparent keyword-based diagnostics; this is not a vendor ATS score."""
    items = ats_analysis.get("top_keywords", [])

    vacancy_keywords = [
        item.get("keyword", "").strip()
        for item in items
        if item.get("keyword", "").strip()
    ]

    supported_keywords = [
        item.get("keyword", "").strip()
        for item in items
        if item.get("status") in {"matched", "partial"} and item.get("keyword", "").strip()
    ]

    missing_keywords = [
        item.get("keyword", "").strip()
        for item in items
        if item.get("status") == "missing" and item.get("keyword", "").strip()
    ]

    vacancy_used = [k for k in vacancy_keywords if _keyword_present(k, resume_text)]
    supported_used = [k for k in supported_keywords if _keyword_present(k, resume_text)]
    unsupported_hits = [k for k in missing_keywords if _keyword_present(k, resume_text)]

    vacancy_total = len(vacancy_keywords)
    supported_total = len(supported_keywords)

    return {
        "vacancy_keyword_coverage_pct": round(100 * len(vacancy_used) / vacancy_total, 1) if vacancy_total else 0.0,
        "supported_keyword_usage_pct": round(100 * len(supported_used) / supported_total, 1) if supported_total else 0.0,
        "vacancy_keywords_used": len(vacancy_used),
        "vacancy_keywords_total": vacancy_total,
        "supported_keywords_used": len(supported_used),
        "supported_keywords_total": supported_total,
        "missing_gap_count": len(missing_keywords),
        "unsupported_keyword_risk_count": len(unsupported_hits),
        "unsupported_keyword_hits": unsupported_hits,
    }


metrics_v1 = calculate_ats_metrics(resume_v1, ats_analysis)
utils.print_html(
    json.dumps(metrics_v1, indent=2, ensure_ascii=False),
    title="ATS Diagnostics (V1)"
)


### 3.4. Step 4 — Reflect on the output with the Employer prompt

The reflection stage keeps the same role as in the original notebook: evaluate V1, identify weaknesses, then regenerate a stronger V2.

Here the reviewer explicitly applies the provided **ROLE PLAY (Employer)** pattern:

> take the role of a recruiter/employer for the target vacancy, identify strengths and gaps, then improve the resume.

The review also revisits **REVISE**, **ACTION VERBS**, **KEYWORDS**, **SKILL COMPARISON**, and **SKILL MATCH** so the second draft can correct weak bullets, missed supported keywords, and unclear skill emphasis.

The reviewer receives the original resume, vacancy, ATS/career-advisor analysis, V1 diagnostics, and Resume V1. Unsupported claims must be removed rather than “optimized.”

In [ ]:
def reflect_on_resume_and_regenerate(
    resume_v1: str,
    original_resume: str,
    vacancy_text: str,
    ats_analysis: dict,
    metrics_v1: dict,
    model_name: str,
) -> tuple[str, str]:
    """
    Apply the Employer/Recruiter role-play prompt to Resume V1,
    then return feedback plus a refined <resume>...</resume> block.
    """

    prompt = f"""
    ROLE PLAY (Employer):
    Take the role of a recruiter/employer hiring for the position described below.
    Review RESUME V1 as if you were screening this candidate for the vacancy.

    ORIGINAL RESUME:
    {original_resume}

    JOB DESCRIPTION:
    {vacancy_text}

    ATS / CAREER ADVISOR ANALYSIS:
    {json.dumps(ats_analysis, indent=2, ensure_ascii=False)}

    V1 ATS DIAGNOSTICS:
    {json.dumps(metrics_v1, indent=2, ensure_ascii=False)}

    RESUME V1:
    {resume_v1}

    Evaluate Resume V1 using these prompt patterns:

    ROLE PLAY (Employer):
    Identify the candidate's strongest relevant evidence, important gaps in skills/experience,
    and anything that would make the resume harder to screen for this vacancy.

    KEYWORDS:
    Identify important supported ATS keywords from the analysis that are missing, weakly placed,
    or overused in Resume V1.

    ACTION VERBS:
    Identify weak, vague, repetitive, or passive bullet openings and replace them with stronger
    role-appropriate action verbs only when the new wording remains faithful to the ORIGINAL RESUME.

    REVISE:
    Improve weak bullets using Action + Task + Method/Skill + Result when the ORIGINAL RESUME
    supplies enough evidence. Never invent a result or metric.

    SKILL COMPARISON:
    Check whether the resume clearly reflects the supported overlap between vacancy requirements
    and the candidate's actual experience.

    SKILL MATCH:
    Check whether the most relevant existing skills and experiences are easy for a recruiter and ATS
    to find. Improve emphasis and placement where useful.

    OUTPUT FORMAT (STRICT):
    1) First line: one valid JSON object with these fields:
       {{
         "feedback": "short overall recruiter critique",
         "strengths": [],
         "gaps": [],
         "weak_or_missing_supported_keywords": [],
         "action_verb_issues": [],
         "bullets_to_revise": [],
         "skill_match_opportunities": [],
         "unsupported_claims": [],
         "recommended_changes": []
       }}

    2) After a newline, output ONLY the improved resume wrapped in:
       <resume>
       ...
       </resume>

    HARD CONSTRAINTS:
    - Do NOT include Markdown, backticks, or extra prose outside the two parts above.
    - Every factual claim in V2 must be supported by the ORIGINAL RESUME.
    - Never add a missing skill, employer, technology, achievement, certification, date, or number.
    - If the vacancy requires something absent from the ORIGINAL RESUME, keep it as a gap.
    - Remove any unsupported keyword or claim introduced in V1.
    - Improve supported ATS keyword usage naturally; do not keyword-stuff.
    - A partial match must remain qualified and must not be rewritten as full expertise.
    - Preserve useful measurable achievements and make bullets concise and specific.
    - Prefer job-description terminology only when it accurately describes existing experience.
    - Keep the result ATS-friendly: plain text sections, simple bullets, no tables or graphics.
    """

    content = utils.get_response(model_name, prompt).strip()

    # Parse the first JSON object for reflection feedback
    first_line = content.splitlines()[0].strip() if content else ""
    try:
        feedback_obj = json.loads(first_line)
    except Exception:
        m_json = re.search(r"\{[\s\S]*?\}\s*(?=<resume>)", content)
        if m_json:
            try:
                feedback_obj = json.loads(m_json.group(0).strip())
            except Exception as e:
                feedback_obj = {"feedback": f"Failed to parse reflection JSON: {e}"}
        else:
            feedback_obj = {"feedback": "Reflection JSON was not found."}

    # Extract refined resume
    m_resume = re.search(r"<resume>([\s\S]*?)</resume>", content)
    if not m_resume:
        raise ValueError("No refined <resume>...</resume> block found in reflection output.")

    refined_resume = "<resume>\n" + m_resume.group(1).strip() + "\n</resume>"
    feedback = json.dumps(feedback_obj, indent=2, ensure_ascii=False)
    return feedback, refined_resume

### 3.5 Step 5 — Generate and inspect the improved version (V2)

Run the reflection stage, extract V2, and calculate the same diagnostics again.


In [ ]:
feedback, resume_v2_tagged = reflect_on_resume_and_regenerate(
    resume_v1=resume_v1,
    original_resume=resume_text,
    vacancy_text=vacancy_text,
    ats_analysis=ats_analysis,
    metrics_v1=metrics_v1,
    model_name="auto",
)

utils.print_html(feedback, title="Feedback on Resume V1")
utils.print_html(resume_v2_tagged, title="Regenerated Resume Output (V2)")

resume_v2 = extract_resume(resume_v2_tagged)
metrics_v2 = calculate_ats_metrics(resume_v2, ats_analysis)

utils.print_html(resume_v2, title="Extracted Resume (V2)")
utils.print_html(
    json.dumps(metrics_v2, indent=2, ensure_ascii=False),
    title="ATS Diagnostics (V2)"
)


### 3.6 Step 6 — Visualize the ATS diagnostics

The charts are diagnostic aids, not claims about the score of a particular commercial ATS product.

The first chart compares V1 and V2 coverage. The second summarizes how the vacancy keywords were classified from the original resume evidence.


In [ ]:
def generate_ats_charts(
    metrics_v1: dict,
    metrics_v2: dict,
    ats_analysis: dict,
    comparison_path: str = "ats_comparison.png",
    status_path: str = "keyword_status.png",
) -> tuple[str, str]:
    """Create two simple matplotlib charts for ATS diagnostics."""

    # Chart 1: V1 vs V2 percentage diagnostics
    labels = ["Vacancy keyword coverage", "Supported keyword usage"]
    v1_values = [
        metrics_v1["vacancy_keyword_coverage_pct"],
        metrics_v1["supported_keyword_usage_pct"],
    ]
    v2_values = [
        metrics_v2["vacancy_keyword_coverage_pct"],
        metrics_v2["supported_keyword_usage_pct"],
    ]

    x = range(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar([i - width / 2 for i in x], v1_values, width, label="V1")
    ax.bar([i + width / 2 for i in x], v2_values, width, label="V2")
    ax.set_title("ATS Alignment Diagnostics: V1 vs V2")
    ax.set_ylabel("Coverage (%)")
    ax.set_ylim(0, 100)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=10, ha="right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(comparison_path, dpi=300)
    plt.close(fig)

    # Chart 2: evidence status of vacancy keywords
    status_counts = {"matched": 0, "partial": 0, "missing": 0}
    for item in ats_analysis.get("top_keywords", []):
        status = item.get("status")
        if status in status_counts:
            status_counts[status] += 1

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(list(status_counts.keys()), list(status_counts.values()))
    ax.set_title("Vacancy Keyword Evidence in Original Resume")
    ax.set_xlabel("Evidence status")
    ax.set_ylabel("Number of keywords")
    fig.tight_layout()
    fig.savefig(status_path, dpi=300)
    plt.close(fig)

    return comparison_path, status_path


comparison_chart, status_chart = generate_ats_charts(
    metrics_v1,
    metrics_v2,
    ats_analysis,
)

utils.print_html(comparison_chart, title="ATS Comparison: V1 vs V2", is_image=True)
utils.print_html(status_chart, title="Keyword Evidence Status", is_image=True)


### 3.7 Optional — Inspect keyword evidence as a table

This makes the analysis auditable: you can see which keyword came from the vacancy, how it was classified, and what evidence the original resume contained.


In [ ]:
def ats_analysis_to_dataframe(ats_analysis: dict) -> pd.DataFrame:
    rows = []
    for item in ats_analysis.get("top_keywords", []):
        rows.append({
            "keyword": item.get("keyword", ""),
            "importance": item.get("importance", ""),
            "status": item.get("status", ""),
            "evidence": item.get("evidence", ""),
        })
    return pd.DataFrame(rows)

keyword_df = ats_analysis_to_dataframe(ats_analysis)
utils.print_html(keyword_df, title="ATS Keyword Evidence Table")


## 4. Put it all together — creating the end-to-end workflow

As in the original notebook, the final `run_workflow` function connects the same individual stages and executes them immediately in sequence:

1. Analyze the vacancy against the original resume using **KEYWORDS + Career Advisor + Skill Comparison + Skill Match + Action Verbs**.
2. Generate tailored Resume V1 using **REVISE + Action Verbs + Keywords + Skill Comparison + Skill Match**.
3. Extract V1 and calculate transparent diagnostics.
4. Reflect on V1 using the explicit **ROLE PLAY (Employer/Recruiter)** prompt plus the same improvement patterns.
5. Generate and extract V2.
6. Calculate V2 diagnostics.
7. Create V1-vs-V2 and keyword-status charts.

The orchestration itself is intentionally unchanged so you can compare prompt quality without changing the workflow architecture.

In [ ]:
def run_workflow(
    resume_text: str,
    vacancy_text: str,
    generation_model: str,
    reflection_model: str,
    image_basename: str = "ats_resume",
):
    """
    End-to-end ATS resume reflection pipeline:
      1) analyze vacancy vs original resume
      2) generate Resume V1
      3) extract V1 + calculate diagnostics
      4) reflect on V1 → feedback + Resume V2
      5) extract V2 + calculate diagnostics
      6) generate comparison charts

    Returns a dict with all artifacts.
    """

    # 0) Display inputs
    utils.print_html(resume_text, title="Original Resume")
    utils.print_html(vacancy_text, title="Target Vacancy")

    # 1) ATS analysis
    utils.print_html("Step 1: Analyzing vacancy and resume for ATS alignment…")
    ats_analysis = analyze_ats_match(
        resume_text=resume_text,
        vacancy_text=vacancy_text,
        model=generation_model,
    )
    utils.print_html(
        json.dumps(ats_analysis, indent=2, ensure_ascii=False),
        title="ATS Analysis",
    )

    # 2) Generate Resume V1
    utils.print_html("Step 2: Generating tailored resume (V1)…")
    resume_v1_tagged = generate_resume(
        resume_text=resume_text,
        vacancy_text=vacancy_text,
        ats_analysis=ats_analysis,
        model=generation_model,
    )
    utils.print_html(resume_v1_tagged, title="LLM Output with Resume V1")

    resume_v1 = extract_resume(resume_v1_tagged)
    utils.print_html(resume_v1, title="Tailored Resume (V1)")

    # 3) Calculate V1 diagnostics
    utils.print_html("Step 3: Calculating transparent ATS diagnostics for V1…")
    metrics_v1 = calculate_ats_metrics(resume_v1, ats_analysis)
    utils.print_html(
        json.dumps(metrics_v1, indent=2, ensure_ascii=False),
        title="ATS Diagnostics (V1)",
    )

    # 4) Reflect and regenerate V2
    utils.print_html("Step 4: Reflecting on Resume V1 and generating improvements…")
    feedback, resume_v2_tagged = reflect_on_resume_and_regenerate(
        resume_v1=resume_v1,
        original_resume=resume_text,
        vacancy_text=vacancy_text,
        ats_analysis=ats_analysis,
        metrics_v1=metrics_v1,
        model_name=reflection_model,
    )
    utils.print_html(feedback, title="Reflection Feedback on V1")
    utils.print_html(resume_v2_tagged, title="LLM Output with Resume V2")

    # 5) Extract V2 and calculate diagnostics
    utils.print_html("Step 5: Extracting Resume V2 and recalculating diagnostics…")
    resume_v2 = extract_resume(resume_v2_tagged)
    metrics_v2 = calculate_ats_metrics(resume_v2, ats_analysis)
    utils.print_html(resume_v2, title="Tailored Resume (V2)")
    utils.print_html(
        json.dumps(metrics_v2, indent=2, ensure_ascii=False),
        title="ATS Diagnostics (V2)",
    )

    # 6) Generate charts
    utils.print_html("Step 6: Generating ATS comparison charts…")
    comparison_path = f"{image_basename}_comparison.png"
    status_path = f"{image_basename}_keyword_status.png"
    comparison_chart, status_chart = generate_ats_charts(
        metrics_v1=metrics_v1,
        metrics_v2=metrics_v2,
        ats_analysis=ats_analysis,
        comparison_path=comparison_path,
        status_path=status_path,
    )
    utils.print_html(comparison_chart, title="ATS Comparison: V1 vs V2", is_image=True)
    utils.print_html(status_chart, title="Vacancy Keyword Evidence", is_image=True)

    return {
        "ats_analysis": ats_analysis,
        "resume_v1": resume_v1,
        "metrics_v1": metrics_v1,
        "feedback": feedback,
        "resume_v2": resume_v2,
        "metrics_v2": metrics_v2,
        "comparison_chart": comparison_chart,
        "status_chart": status_chart,
    }


### 4.2. Try the workflow

Replace `resume_text` and `vacancy_text` with your actual strings, then run the complete workflow.

As in the original notebook, you can use a faster model for generation and a stronger reasoning model for reflection.


In [ ]:
# Here, insert your updates
user_resume = resume_text
user_vacancy = vacancy_text

generation_model = "auto"
reflection_model = "auto"
# You can also set an explicit Cursor-supported model, e.g. "gpt-5"
image_basename = "ats_resume"

# Run the complete agentic workflow
result = run_workflow(
    resume_text=user_resume,
    vacancy_text=user_vacancy,
    generation_model=generation_model,
    reflection_model=reflection_model,
    image_basename=image_basename,
)


## 5. Final Takeaways

In this lab, **you** reused the original Generate → Parse → Reflect → Regenerate pattern for ATS resume tailoring while explicitly incorporating the supplied resume prompt patterns.

The workflow now operationalizes:

- **ACTION VERBS** — role-appropriate active wording supported by real experience,
- **KEYWORDS** — ATS-relevant terms extracted from the vacancy,
- **ROLE PLAY (Career Advisor)** — priorities for what the candidate should emphasize,
- **ROLE PLAY (Employer)** — recruiter-style critique of Resume V1,
- **REVISE** — targeted bullet and wording improvements,
- **SKILL COMPARISON** — vacancy requirements compared with resume evidence,
- **SKILL MATCH** — supported skills and experiences prioritized for the target role.

The architecture remains the same as the original notebook. Only the prompts are richer, which makes this version suitable for a direct comparison against the previous ATS notebook.

The key safety constraint also remains unchanged: **never improve apparent ATS alignment by inventing candidate experience**.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Congratulations!</strong>

You’ve completed the lab on building an **agentic ATS resume-tailoring workflow**.

The workflow preserves the original notebook pattern while replacing chart refinement with evidence-aware resume refinement and ATS diagnostics.

</div>
